# Extent, $\gamma$, $\omega$ errors, miss fraction and run time against NBASIS per WIDTH

Loads, for the four `kbm_free` Latin hypercubes (SPR-045, M1, R4, R3):

- the GS2 `pyroscan.nc` under `$GK_DATA_ROOT/GS2/Runs/LATIN_HYPERCUBE/<case>/kbm_free`;
- the TGLF leaves `$GK_DATA_ROOT/TGLF/Runs/LATIN_HYPERCUBE/<case>/kbm_free/fx_<filter>_<w>_nb<N>` (SAT_RULE 2, `FIND_WIDTH` F,
  NBASIS_MIN = NBASIS_MAX = $N$, NXGRID 32, `USE_MHD_RULE` F, `USE_BPER` T, `USE_BPAR` T, IBRANCH $-1$), $N \in \{2,4,6,8,12,16,24\}$;
- the GFTM leaves `.../GFTM/Runs/LATIN_HYPERCUBE/<case>/kbm_free/galF[_f0p0]_<w>_nb<N>_nx<X>` (FIELD_GALERKIN F; FILTER 0.5, or 0 for `f0p0`),
  $(N, X)$ = (2,32), (4,32), (8,32), (12,32), (16,32), (24,32), (32,40), (48,65), (56,64), plus (32,65) on a 50-sample subset.

at the 24 widths 0.2 to 4.0. Every cell has NXGRID $\ge$ NBASIS + 8.

Per sample: $r_\gamma=\log_{10}(\gamma/\gamma_{\rm GS2})$, $r_\omega=\log_{10}(\omega/\omega_{\rm GS2})$, and $r_\phi$, $r_{A_\parallel}$, the
$\log_{10}$ ratio of pyro's 95% `Extent` of $|\phi|^2$ and $|A_\parallel|^2$, from the dominant mode (mode 0, checked against
`argmax("mode")`). A sample is a **miss** if it has no unstable mode or its dominant mode is in the electron direction. Each
(code, FILTER, NBASIS, NXGRID, WIDTH) group is reduced to the RMS of $r$ over non-misses with a bootstrap 95% CI, the median $|r|$
and the miss fraction.

Plots, per code, against NBASIS with one line per WIDTH: RMS $r_\gamma$, $r_\omega$, $r_\phi$, $r_{A_\parallel}$, the miss fraction and the wall
time per run.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"
import json
import re
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator, StrMethodFormatter
import pyrokinetics
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)

data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # WORK: the GS2 kbm_free cubes and the leaves
run_root = data_root     # the TGLF/GFTM leaves
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "kbm_fixed_width_nbasis"
csv_dir = Path("csv")
plots_dir.mkdir(parents=True, exist_ok=True); csv_dir.mkdir(exist_ok=True)
cases = ["SPR-045", "M1", "R4", "R3"]
widths = [0.2, 0.2297, 0.2639, 0.3031, 0.3482, 0.4, 0.4595, 0.5279, 0.6063, 0.6965, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6,
          1.9218, 2.3083, 2.7726, 3.3302, 4.0]    # registry.WIDTH_ORACLE_GRID
width_tag = lambda width: f"w{width:.4f}".replace(".", "p")
tglf_nbasis = [2, 4, 6, 8, 12, 16, 24]    # TGLF NXGRID 32
gftm_cells = {2: 32, 4: 32, 8: 32, 12: 32, 16: 32, 24: 32, 32: 40, 48: 65, 56: 64}   # NBASIS: NXGRID, all with NXGRID >= NBASIS + 8
tglf_filters = {"f0p0": 0.0, "f2p0": 2.0, "fm1p5": -1.5}
gftm_filters = {"": 0.5, "_f0p0": 0.0}
tglf_filter_tag = os.environ.get("TGLF_FILTER_TAG", "fm1p5")    # the Part 0 winner, used for Parts A and B
metrics = ["gamma", "omega", "phi", "apar"]
metric_label = {"gamma": r"$\gamma$", "omega": r"$\omega$", "phi": r"$\phi$ extent", "apar": r"$A_\parallel$ extent"}
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
rng = np.random.default_rng(0)

## GS2 reference
Loads each `kbm_free` GS2 cube, applies the species ties to every sample, labels samples with `mode_classification.classify`
(tearing parameter, frequency direction, $\chi_e/\chi_i$) and keeps KBMs with $\gamma>0$ and `growth_rate_tolerance` $<0.05$
(738 pooled). Stores each one's $\gamma$, $\omega$ and 95% extents of $|\phi|^2$, $|A_\parallel|^2$; the GS2 $\theta$ grid is NaN-padded per
sample, so the extent is taken on the finite points.

In [ ]:
def tied_species(sample):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share ion1's a/L_T, every a/L_n is the electron's
    species = gs2_scan.sample_pyro(sample).local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")
    return {name: (float(species[name].temp.m), float(species[name].dens.m), float(species[name].inverse_lt.m)) for name in species.names}


def extent_rows(power):
    # pyro's 95% rule per sample on its own finite theta points; power: (sample, theta) DataArray
    if np.isfinite(power.values).all():
        return Extent._compute(power, 0.95)[0].values
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if np.isfinite(row.values).any() else np.nan
                     for row in power])


gs2 = {}
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    with get_context("fork").Pool(n_cpu) as pool:
        species = pool.map(tied_species, range(len(names)), chunksize=8)
    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()

    def chi(group):   # (Q_s - 1.5 T_s Gamma_s) / (n_s T_s a/L_Ts) summed over the group, as mode_classification.indicators
        numerator = denominator = 0.0
        for name in group:
            temp, dens, inverse_lt = np.array([sp[name] for sp in species]).T
            numerator = numerator + heat.sel(species=name).values - 1.5 * temp * particle.sel(species=name).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    with np.errstate(divide="ignore", invalid="ignore"):
        labels = np.asarray(mc.classify(dict(T=gs2_output.tearing_parameter.values.ravel(),
                                             omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                             chi_ratio=chi(["electron"]) / chi(ions))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    kbm = np.isfinite(gamma) & (gamma > 0) & (tolerance < 0.05) & (labels == "KBM")
    selected = gs2_output.set_index(sample="sample_name").sel(sample=names[kbm])
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=mc.ION_DIRECTION * selected.mode_frequency.pint.dequantify().values.ravel(),
                     phi=extent_rows(selected.phi_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     apar=extent_rows(selected.apar_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     units=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)))
    print(case, "GS2-KBMs", kbm.sum(), "| median GS2 extent phi", np.median(gs2[case]["phi"]), "apar", np.median(gs2[case]["apar"]), flush=True)
    del gs2_output, selected
reference = pd.concat([pd.DataFrame(dict(case=case, sample=gs2[case]["names"], **{f"{m}_gs2": gs2[case][m] for m in metrics})) for case in cases])
assert len(reference) == 738, len(reference)

## TGLF and GFTM leaves
Loads one `PyroHypercube` per leaf and checks its `manifest.json` `code_flags` against the leaf name. Takes the dominant mode's
$\gamma$, $\omega$ (ion direction positive) and extents, and counts over all modes the unstable ones, those with odd $\phi$
(`phi_even_fraction` $<0.5$) and those with even $A_\parallel$ (`apar_even_fraction` $>0.5$). Joins to the GS2 table by sample and
forms the log ratios and group summaries.

In [ ]:
def leaves():
    for case in cases:
        for tag, value in tglf_filters.items():
            for nbasis in tglf_nbasis:
                for width in widths:
                    yield "TGLF", case, value, nbasis, 32, width, run_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"fx_{tag}_{width_tag(width)}_nb{nbasis}"
        for tag, value in gftm_filters.items():
            for nbasis, nxgrid in [*gftm_cells.items(), (32, 65)]:
                for width in widths:
                    yield "GFTM", case, value, nbasis, nxgrid, width, run_root / "GFTM/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"galF{tag}_{width_tag(width)}_nb{nbasis}_nx{nxgrid}"


def load_leaf(spec):
    code, case, filter_value, nbasis, nxgrid, width, leaf = spec
    if not (leaf / "pyroscan.nc").is_file():
        return None
    scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]
    assert (float(flags["WIDTH"]), float(flags["FILTER"]), int(flags["NBASIS_MAX"]), int(flags["NXGRID"])) == (width, filter_value, nbasis, nxgrid), leaf
    assert not flags["FIND_WIDTH"] and (code == "GFTM" or (flags["SAT_RULE"] == 2 and not flags["USE_MHD_RULE"] and flags["USE_BPAR"])), leaf
    output = scan.gk_output.data
    assert re.sub(r"_pyroscan_base\d+", "", str(output.growth_rate.data.units)) == gs2[case]["units"], "normalisations differ"
    names = [str(name) for name in output.sample_name.values]
    output = output.set_index(sample="sample_name")
    growth = output.growth_rate.pint.dequantify().transpose("sample", "mode")
    assert (growth.fillna(-np.inf).argmax("mode") == 0).all(), leaf     # IBRANCH -1: mode 0 is the dominant mode
    frequency = mc.ION_DIRECTION * output.mode_frequency.pint.dequantify().transpose("sample", "mode")
    unstable = growth.fillna(0).values > 0
    row = dict(code=code, case=case, filter=filter_value, nbasis=nbasis, nxgrid=nxgrid, width=width, sample=names,
               failed=growth.isnull().all("mode").values, gamma=growth.isel(mode=0).fillna(0).values, omega=frequency.isel(mode=0).values, n_unstable=unstable.sum(1))  # mode0-guard-allow: IBRANCH -1, argmax==0 asserted above
    for field in ("phi", "apar"):
        power = (np.abs(output[field].isel(mode=0).pint.dequantify().squeeze("kx")) ** 2).transpose("sample", "theta")  # mode0-guard-allow: as above
        row[field] = np.where(row["gamma"] > 0, extent_rows(power), np.nan)
    if "phi_even_fraction" in output:
        row["n_unstable_phi_odd"] = (unstable & (output.phi_even_fraction.pint.dequantify().squeeze().transpose("sample", "mode").values < 0.5)).sum(1)
        row["n_unstable_apar_tearing"] = (unstable & (output.apar_even_fraction.pint.dequantify().squeeze().transpose("sample", "mode").values > 0.5)).sum(1)
    return pd.DataFrame(row)


with get_context("fork").Pool(n_cpu) as pool:
    runs = pd.concat([table for table in pool.imap_unordered(load_leaf, list(leaves()), chunksize=4) if table is not None], ignore_index=True)
runs = runs.merge(reference, on=["case", "sample"], validate="many_to_one")
print("runs with no output (all modes NaN), dropped:", runs.failed.sum(), "of", len(runs), flush=True)
if runs.failed.any():
    print(runs[runs.failed].groupby(["code", "case", "filter", "nbasis", "nxgrid"]).size().to_string(), flush=True)
runs = runs[~runs.failed]
assert (runs.nxgrid >= runs.nbasis + 8).all()
runs["miss"] = ~((runs.gamma > 0) & (runs.omega > 0))
for metric in metrics:
    runs[f"r_{metric}"] = np.where(runs.miss, np.nan, np.log10(runs[metric] / runs[f"{metric}_gs2"]))
print(runs.groupby(["code", "filter", "nxgrid", "nbasis"]).agg(leaves=("width", "nunique"), rows=("sample", "size")).to_string(), flush=True)


def summarise(group):
    # RMS of r over non-misses, its bootstrap 95% CI, median |r|, miss fraction
    out = dict(n=len(group), miss_fraction=group.miss.mean())
    for metric in metrics:
        values = group[f"r_{metric}"].dropna().values
        boot = np.sqrt((rng.choice(values, (1000, values.size)) ** 2).mean(1)) if values.size else np.full(1, np.nan)
        out.update({f"{metric}_rms": np.sqrt((values ** 2).mean()) if values.size else np.nan, f"{metric}_lo": np.percentile(boot, 2.5),
                    f"{metric}_hi": np.percentile(boot, 97.5), f"{metric}_median_abs": np.median(np.abs(values)) if values.size else np.nan})
    return pd.Series(out)


keys = ["code", "filter", "nbasis", "nxgrid", "width"]
argmin_width = lambda table, column: table.sort_values("width").pipe(lambda d: d.width.values[np.nanargmin(d[column].values)] if np.isfinite(d[column].values).any() else np.nan)
pooled = runs.groupby(keys).apply(summarise).reset_index()
per_case = runs.groupby(["case", *keys]).apply(summarise).reset_index()

## Select widths
Takes TGLF at the FILTER in `TGLF_FILTER_TAG` (default $-1.5$) and GFTM at FILTER 0.5 on the listed (NBASIS, NXGRID) cells. Collects
the widths minimising each metric at the top NBASIS (and $\gamma$ at NBASIS 4), merges neighbouring grid points, adds the extreme
per-case $\gamma$ minima if fewer than three remain, and writes those widths' rows to `candidate_widths_by_nbasis.csv`.

In [ ]:
arm_filter = {"TGLF": tglf_filters[tglf_filter_tag], "GFTM": 0.5}
arm = pooled[[(row.filter == arm_filter[row.code]) and row.nxgrid >= row.nbasis + 8 and (row.code == "TGLF" or (row.nbasis, row.nxgrid) in gftm_cells.items()) for row in pooled.itertuples()]]
arm_filter = {code: value for code, value in arm_filter.items() if (arm.code == code).any()}   # a code whose leaves are not run yet is skipped
top_nbasis = {code: arm[arm.code == code].nbasis.max() for code in arm_filter}
candidates = {}
for code in arm_filter:
    data = arm[arm.code == code]
    argmins = {metric: argmin_width(data[data.nbasis == top_nbasis[code]], f"{metric}_rms") for metric in metrics}
    argmins["gamma_nb4"] = argmin_width(data[data.nbasis == 4], "gamma_rms")
    merged = []
    for width in sorted({width for width in argmins.values() if np.isfinite(width)}):
        if not merged or widths.index(width) - widths.index(merged[-1][-1]) > 1:
            merged.append([width])
        else:
            merged[-1].append(width)
    candidates[code] = [group[len(group) // 2] for group in merged]
    per_case_argmin = per_case[(per_case.code == code) & (per_case.nbasis == top_nbasis[code]) & (per_case["filter"] == arm_filter[code])
                               & (per_case.nxgrid == (32 if code == "TGLF" else gftm_cells[top_nbasis[code]]))].groupby("case").apply(
        lambda d: pd.Series({metric: argmin_width(d, f"{metric}_rms") for metric in metrics}))
    print(code, "per-case argmin at the top NBASIS:\n", per_case_argmin.to_string())
    if len(candidates[code]) < 3:   # pooled argmins coincide: bracket them with the per-case gamma argmins' extremes
        for width in (per_case_argmin.gamma.min(), per_case_argmin.gamma.max()):
            if min(abs(widths.index(width) - widths.index(c)) for c in candidates[code]) > 1:
                candidates[code].append(width)
        candidates[code].sort()
    print(code, "argmins:", argmins, "-> candidates", candidates[code])

candidate_table = pd.concat([arm[(arm.code == code) & arm.width.isin(candidates[code])] for code in arm_filter])
candidate_table.to_csv(csv_dir / "candidate_widths_by_nbasis.csv", index=False)

## Plot against NBASIS per WIDTH
Plots RMS $r$ with CI bars and the miss fraction against NBASIS at the selected widths (GFTM also at 0.3482, 0.4595, 0.5279, 0.6063,
0.8 and 1.1), and the median and interquartile wall time per run from `$TIMING_CSV` and `$TIMING_NX8_CSV` (a power law is fitted for
$N \ge 8$ and printed). For each width, `nbasis_converged` is the smallest $N$ from which every larger $N$ keeps all four RMS values
inside the top-$N$ CI. Writes `timing_<code>.csv`, `recommendation.csv`, `per_case_candidates.csv` and `pooled_summary.csv`.

In [ ]:
timing_file = Path(os.environ.get("TIMING_CSV", "timing.csv"))
timing = pd.read_csv(timing_file) if timing_file.is_file() else None
timing_nx8_file = Path(os.environ.get("TIMING_NX8_CSV", "timing_nx8.csv"))   # the (32, 40) and (56, 64) pairs, same protocol
if timing is not None and timing_nx8_file.is_file():
    timing = pd.concat([timing, pd.read_csv(timing_nx8_file)])
if timing is not None:
    timing = timing[(timing.nxgrid >= timing.nbasis + 8)]
partB_widths = {code: sorted(set(candidates[code]) | ({0.3482, 0.4595, 0.5279, 0.6063, 0.8, 1.1} if code == "GFTM" else set())) for code in arm_filter}
recommendation = []
for code in arm_filter:
    data = arm[arm.code == code]
    fig, axes = plt.subplots(1, 6, figsize=(36, 6), sharex=True, constrained_layout=True)
    for width in partB_widths[code]:
        curve = data[data.width == width].sort_values("nbasis")
        top = curve[curve.nbasis == top_nbasis[code]].iloc[0]
        for ax, metric in zip(axes, metrics):
            ax.errorbar(curve.nbasis, curve[f"{metric}_rms"], yerr=[curve[f"{metric}_rms"] - curve[f"{metric}_lo"], curve[f"{metric}_hi"] - curve[f"{metric}_rms"]],
                        fmt="o-", capsize=3, label=f"WIDTH {width:g}")
        axes[4].plot(curve.nbasis, curve.miss_fraction, "o-")
        # inside the top-N bootstrap CI for every metric, at this N and every larger one (an RMS BELOW the CI is not convergence)
        inside = np.all([(curve[f"{metric}_rms"] >= top[f"{metric}_lo"]) & (curve[f"{metric}_rms"] <= top[f"{metric}_hi"]) for metric in metrics], axis=0)
        stays = np.cumprod(inside[::-1])[::-1].astype(bool)
        nbasis_converged = int(curve.nbasis.values[stays].min())
        at = curve[curve.nbasis == nbasis_converged]
        recommendation.append(dict(code=code, width=width, nbasis_converged=nbasis_converged,
                                   **{f"{metric}_rms_at_converged": at[f"{metric}_rms"].iloc[0] if len(at) else np.nan for metric in metrics},
                                   **{f"{metric}_rms_top": top[f"{metric}_rms"] for metric in metrics}, miss_at_converged=at.miss_fraction.iloc[0] if len(at) else np.nan))
    if timing is not None and (timing.code == code).any():
        times = timing[timing.code == code].groupby("nbasis").seconds.describe()
        axes[5].errorbar(times.index, times["50%"], yerr=[times["50%"] - times["25%"], times["75%"] - times["50%"]], fmt="o-", capsize=3)
        exponent, _ = np.polyfit(np.log(times.index[times.index >= 8]), np.log(times["50%"][times.index >= 8]), 1)
        axes[5].set_title(f"{code}, cost", fontsize=16)   # the fitted exponent goes in the README
        print(code, "timing\n", times[["count", "25%", "50%", "75%"]].round(3).to_string(), "\n exponent", round(exponent, 2))
        times.to_csv(csv_dir / f"timing_{code}.csv")
        axes[5].set_yscale("log")
    for ax, metric in zip(axes, metrics):
        ax.set_title(f"{code}, {metric_label[metric]}", fontsize=16); ax.set_yscale("log")
        ax.set_ylabel(f"RMS $\\log_{{10}}$ ratio to GS2", fontsize=16)
    axes[4].set_title(f"{code}, misses", fontsize=16); axes[4].set_ylabel("fraction", fontsize=16)
    axes[5].set_ylabel("wall time per run [s]", fontsize=16)
    for ax in axes:
        ax.set_xscale("log", base=2); ax.set_xlabel("NBASIS", fontsize=16); ax.tick_params(axis="both", labelsize=14)
        ax.set_xticks(sorted(data.nbasis.unique())); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())   # keep the log-y minor labels: most panels span under a decade
    axes[0].legend(fontsize=12)
    fig.savefig(plots_dir / f"{code}_errors_and_no_ion_mode_fraction_and_run_time_vs_nbasis.png", dpi=150); fig.savefig(plots_dir / f"{code}_errors_and_no_ion_mode_fraction_and_run_time_vs_nbasis.pdf")
recommendation = pd.DataFrame(recommendation)
print(recommendation.round(3).to_string())
recommendation.to_csv(csv_dir / "recommendation.csv", index=False)
per_case[per_case.width.isin([w for c in candidates.values() for w in c])].to_csv(csv_dir / "per_case_candidates.csv", index=False)
pooled.to_csv(csv_dir / "pooled_summary.csv", index=False)